In [0]:
display(
    spark.sql("""
        select
            current_user() as current_user,
            current_catalog() as current_catalog,
            current_schema() as current_schema,
            current_timestamp() as executed_at_utc
    """)
)

In [0]:
catalog = "workspace"
schema = "gtfs_bogota"
volume = "raw_gtfs"

raw_volume_path = f"/Volumes/{catalog}/{schema}/{volume}"

spark.sql(f"use catalog `{catalog}`")
spark.sql(f"use schema `{schema}`")

print(f"Raw Volume: {raw_volume_path}")
display(dbutils.fs.ls(raw_volume_path))

In [0]:
# BLOQUE 3 — Conectividad resiliente y metadata de la corrida

from datetime import datetime, timezone
import hashlib
import json
import time
import uuid

import requests

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    LongType,
)

# 1. Metadata de nuestra ejecución
run_id = str(uuid.uuid4())
captured_at_utc = datetime.now(timezone.utc)

# 2. Fuente pública que queremos consultar
source_url = "https://gis.transmilenio.gov.co/gtfs/vehiclepos.pb"

# 3. Política de reintentos
max_attempts = 3
retryable_status_codes = {429, 500, 502, 503, 504}

response = None
attempts = []

# 4. Petición HTTP con reintentos controlados
for attempt_number in range(1, max_attempts + 1):
    try:
        response = requests.get(
            source_url,
            headers={
                "User-Agent": "bogota-gtfs-mobility-lakehouse/0.1"
            },
            timeout=(10, 30),
        )

        attempts.append({
            "attempt_number": attempt_number,
            "http_status": response.status_code,
            "error": None,
        })

        # Éxito: detenemos los reintentos
        if response.status_code == 200:
            break

        # Solo reintentamos errores que normalmente son temporales
        should_retry = (
            response.status_code in retryable_status_codes
            and attempt_number < max_attempts
        )

        if should_retry:
            wait_seconds = 2 ** (attempt_number - 1)

            print(
                f"Intento {attempt_number} respondió HTTP "
                f"{response.status_code}. Reintentando en "
                f"{wait_seconds} segundos..."
            )

            time.sleep(wait_seconds)

    except requests.RequestException as error:
        attempts.append({
            "attempt_number": attempt_number,
            "http_status": None,
            "error": str(error),
        })

        if attempt_number < max_attempts:
            wait_seconds = 2 ** (attempt_number - 1)

            print(
                f"Intento {attempt_number} falló por red. "
                f"Reintentando en {wait_seconds} segundos..."
            )

            time.sleep(wait_seconds)

# 5. Solo calculamos hash si recibimos un feed válido
if response is not None and response.status_code == 200:
    content = response.content
    content_hash = hashlib.sha256(content).hexdigest()
    bytes_received = len(content)
else:
    content = None
    content_hash = None
    bytes_received = 0

# 6. Esquema explícito: Spark no necesita adivinar tipos
result_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("captured_at_utc", StringType(), False),
    StructField("source_url", StringType(), False),
    StructField("final_http_status", IntegerType(), True),
    StructField("bytes_received", LongType(), False),
    StructField("sha256", StringType(), True),
    StructField("attempt_count", IntegerType(), False),
    StructField("attempts_json", StringType(), False),
])

# 7. Una fila con el resultado de toda la corrida
result_row = [(
    run_id,
    captured_at_utc.isoformat(),
    source_url,
    response.status_code if response is not None else None,
    bytes_received,
    content_hash,
    len(attempts),
    json.dumps(attempts),
)]

# 8. Mostrar resultado en Databricks
result_df = spark.createDataFrame(
    result_row,
    schema=result_schema,
)

display(result_df)

In [0]:
import json
import os
from pathlib import Path

if response.status_code != 200:
    raise RuntimeError(
        f"No se guarda un snapshot porque el endpoint respondió HTTP "
        f"{response.status_code}."
    )

feed_name = "vehicle_positions"
file_name = "vehiclepos.pb"

snapshot_dir = (
    f"{raw_volume_path}/{feed_name}/snapshots/"
    f"sha256={result['sha256']}"
)

raw_target_path = f"{snapshot_dir}/{file_name}"

ingest_date = captured_at_utc.strftime("%Y-%m-%d")

manifest_dir = (
    f"{raw_volume_path}/{feed_name}/manifests/"
    f"ingest_date={ingest_date}"
)

manifest_path = f"{manifest_dir}/run_id={run_id}.json"

dbutils.fs.mkdirs(snapshot_dir)
dbutils.fs.mkdirs(manifest_dir)

existing_files = [file.name for file in dbutils.fs.ls(snapshot_dir)]
snapshot_already_exists = file_name in existing_files

if not snapshot_already_exists:
    local_file_path = f"/tmp/{run_id}_{file_name}"

    with open(local_file_path, "wb") as file:
        file.write(response.content)

    dbutils.fs.cp(
        f"file:{local_file_path}",
        raw_target_path,
    )

    os.remove(local_file_path)

manifest = {
    **result,
    "feed_name": feed_name,
    "raw_snapshot_path": raw_target_path,
    "snapshot_already_exists": snapshot_already_exists,
}

dbutils.fs.put(
    manifest_path,
    json.dumps(manifest, indent=2),
    overwrite=False,
)

display(spark.createDataFrame([manifest]))